# SDS 示波器数据采集示例

本示例演示如何使用 `sds_acquisition` 包控制 SDS 系列数字示波器进行数据采集。

## 前提条件
- 示波器已通过 USB 连接
- VISA 驱动已安装
- 虚拟环境 `agent_exp_env` 已配置好依赖
- 在 VSCode 中打开此 notebook 后，右上角选择内核 `Python (agent_exp_env)`

In [1]:
# 设置项目根目录（在 examples/ 目录中运行）
from pathlib import Path
project_root = Path.cwd().parent

In [2]:
from sds_acquisition import (
    SDSInstrument, SDSAcquisition, AcquisitionConfig,
    save_to_csv, save_to_npz, save_to_mat, save_to_h5, save_to_bin,
    load_npz, load_h5, load_bin,
)
import numpy as np
import matplotlib.pyplot as plt
print("模块导入成功")

模块导入成功


---
## 1. 配置管理

### 1.1 从默认 YAML 文件加载配置

In [3]:
config_path = project_root / "params" / "default_config.yaml"
config = AcquisitionConfig.from_yaml(str(config_path))
print("当前配置:")
print(f"  采样率: {config.sampling_rate/1e3:.2f} KSa/s")
print(f"  采样时间: {config.sampling_time*1e3:.2f} ms")
print(f"  时基 (自动计算): {config.timebase_scale*1e3:.3f} ms/div")
print(f"  总点数 (自动计算): {config.total_points}")
print(f"  触发延时: {config.acquire_delay:.2f} s")
for ch in config.channels:
    if ch.enabled:
        print(f"  CH{ch.number}: {ch.scale} V/div, offset={ch.offset} V, {ch.coupling}")

当前配置:
  采样率: 100.00 KSa/s
  采样时间: 1000.00 ms
  时基 (自动计算): 100.000 ms/div
  总点数 (自动计算): 100000
  触发延时: 1.00 s
  CH1: 0.001 V/div, offset=0.0 V, DC


### 1.2 编程方式创建和修改配置

In [10]:
config2 = AcquisitionConfig()

config2.sampling_rate = 1000e3       # 500 kSa/s
config2.sampling_time = 1           # 1 s 总采集窗口
config2.acquire_delay = 2        # 触发后等待 2 s
config2.acquire_type = "NORMal"

config2.channels[0].enabled = True
config2.channels[0].scale = 0.001
config2.channels[0].offset = 0.0
config2.channels[1].enabled = True
config2.channels[1].scale = 0.001
config2.channels[1].offset = -1.0

config2.trigger.mode = "FTRIG"
config2.trigger.source = "C1"
config2.trigger.level = 0.0

print(f"采样率: {config2.sampling_rate/1e3:.1f} kSa/s")
print(f"采样时间: {config2.sampling_time:.1f} s")
print(f"触发延时: {config2.acquire_delay:.2f} s")

采样率: 1000.0 kSa/s
采样时间: 1.0 s
触发延时: 2.00 s


In [11]:
config2.to_yaml(str(project_root / "params" / "my_experiment_config.yaml"))
print("配置已保存")

配置已保存


---
## 2. 执行数据采集

> **注意**: 以下代码需要连接示波器硬件才能运行。`acquire_delay` 控制触发后的等待时间。

In [12]:
RESOURCE = "USB0::0xF4EC::0x1015::SDSEV82X900704::INSTR"
# config_apply = AcquisitionConfig.from_yaml(str(project_root / "params" / "default_config.yaml"))
config_apply = AcquisitionConfig.from_yaml(str(project_root / "params" / "my_experiment_config.yaml"))
try:
    with SDSInstrument(RESOURCE) as inst:
        print(f"已连接: {inst.idn()}")
        print(f"期望点数: {config_apply.total_points} = {config_apply.sampling_rate/1e3:.0f} kSa/s × {config_apply.sampling_time*1e3:.1f} ms")
        acq = SDSAcquisition(inst)
        results = acq.acquire_all(config_apply)
        print(f"采集完成: {len(results)} 通道")
        for r in results:
            print(f"  CH{r.channel}: {len(r.voltage)} 点 "
                  f"(裁剪至: {config_apply.total_points})")
except Exception as e:
    print(f"采集失败: {e}")

已连接: Siglent Technologies,SDS1204X HD,SDSEV82X900704,6.9.12.1.1.3.8
期望点数: 1000000 = 1000 kSa/s × 1000.0 ms
采集完成: 2 通道
  CH1: 1000000 点 (裁剪至: 1000000)
  CH2: 1000000 点 (裁剪至: 1000000)


---
## 3. 数据保存

`save_to_npz()` 支持 `save_mode` 参数控制存储内容：

| save_mode | 存储内容 | 体积 |
|---|---|---|
| `"raw"` (默认) | raw int16 + 元数据 | ~2 B/点 |
| `"voltage_time"` | voltage + time | ~16 B/点 |
| `"voltage_only"` | 仅 voltage | ~8 B/点 |
| `"all"` | raw + voltage + time | ~18 B/点 |

In [25]:
if 'results' in dir() and results:
    save_to_npz("data/", results, config.to_dict(), save_mode="raw")
    save_to_npz("data/", results, config.to_dict(), save_mode="voltage_time")
    save_to_npz("data/", results, config.to_dict(), save_mode="all")
    save_to_h5("data/", results, config.to_dict())
    save_to_bin("data/", results, config.to_dict())
    for r in results:
        save_to_csv("data/", r)
    print("所有格式已保存")
else:
    print("未采集到数据，跳过保存")

所有格式已保存


---
## 4. 加载并绘制波形

In [ ]:
npz_list = sorted(Path("data/").glob("*.npz"))
if npz_list:
    loaded, _ = load_npz(str(npz_list[-1]))
    print(f"加载 {len(loaded)} 通道")

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    for r in loaded:
        axes[0].plot(r.time * 1e3, r.voltage, label=f"CH{r.channel}", lw=0.8)
        axes[1].plot(r.time * 1e3, r.raw_data, label=f"CH{r.channel}", lw=0.8)
    axes[0].set_ylabel("Voltage (V)"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].set_xlabel("Time (ms)"); axes[1].set_ylabel("ADC Code"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    # 使用 project_root 确保路径正确
    plot_path = project_root / "results" / "waveform_plot.png"
    fig.savefig(str(plot_path), dpi=150, bbox_inches="tight")
    print(f"Plot saved to {plot_path}")
else:
    print("未找到 NPZ 文件")

---
## 5. 模拟数据（无硬件时测试）

In [ ]:
fs = 1e9; duration = 10e-6
t = np.linspace(0, duration, int(fs * duration), endpoint=False)
f_sig = 1e6
ch1 = 2.0 * np.sin(2 * np.pi * f_sig * t) + 0.1 * np.random.randn(len(t))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t * 1e6, ch1, label="CH1 (simulated)", lw=0.8)
ax.set_xlabel("Time (µs)"); ax.set_ylabel("Voltage (V)")
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 6. 无触发采集（AUTO 模式）

```python
config.trigger.mode = "AUTO"
with SDSInstrument(RESOURCE) as inst:
    results = SDSAcquisition(inst).acquire_all(config)
```

---
## 7. 快速提示

- **触发延时**: 调整 `acquire_delay` 控制采集等待时间
- **保存内容**: `save_to_npz(..., save_mode="voltage_time")` 只存 voltage+time
- **平均降噪**: `acquire_type = "AVERage"`, `acquire_type_param = 64`
- **旧数据兼容**: `load_npz()` 支持所有 `save_mode`